In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import joblib

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
print("first five rows of dataset:")
print(data.head())

first five rows of dataset:


NameError: name 'data' is not defined

In [ ]:
print("Data Info:")
print(data.info())

In [ ]:
print("Missing values:")
print(data.isnull().sum())

In [ ]:
print("Basic Statistics:")
print(data.describe())

In [ ]:
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')

In [ ]:
numerical_cols = ['tenure','MonthlyCharges','TotalCharges']

In [ ]:
# Missing values in total charges
print(f"Missing values in total charges:\n{data['TotalCharges'].isnull().sum()}")

In [ ]:
plt.figure(figsize=(15,5))
for i, col in enumerate(numerical_cols, 1):
    plt.subplot(1,3,i)
    sns.histplot(data[col], bins=30, kde=True)
    plt.title(f"{col} Distribution")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15,5))
for i, col in enumerate(numerical_cols, 1):
    plt.subplot(1,3,i)
    sns.boxplot(x='Churn', y=col, data=data)
    plt.title(f"{col} by Churn")
plt.tight_layout()
plt.show()
    

In [ ]:
# Categorical features: Bar plots
catagorical_cols = ['gender','Contract','PaymentMethod','InternetService']
plt.figure(figsize=(15,10))
for i, col in enumerate(catagorical_cols, 1):
    plt.subplot(2,2,i)
    sns.countplot(x=col, hue='Churn', data=data)
    plt.title(f"Churn by {col}")
    plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
data['Churn_Encode'] = data['Churn'].map({'Yes':1, 'No':0})
correlation = data[numerical_cols + ['Churn_Encode']].corr()
plt.figure(figsize=(8,6))
sns.heatmap(correlation, annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()
print("Correlation with Churn\n", correlation['Churn_Encode'])

In [ ]:
items = data[(data['tenure'] <40) & (data['Contract'] == 'Month-to-month')]

In [ ]:
items

In [ ]:
nonchurn = (data['Churn_Encode'] == 1).sum()
churn = (data['Churn_Encode'] == 0).sum()

In [ ]:
print(f"Numbre of churners with less than 40 tanure and Month to month contract, {churn}")
print(f"Numbre of not-churners with less than 40 tanure and Month to month contract, {nonchurn}")

In [ ]:
payment_methods = data['PaymentMethod'].unique()
print(payment_methods)

In [ ]:
churners = ((data['PaymentMethod'] == payment_methods[0]) & (data['Churn_Encode'] == 1)).sum()
non_churners = ((data['PaymentMethod']== payment_methods[0]) & (data['Churn_Encode']== 0)).sum()

churners1 = ((data['PaymentMethod'] == payment_methods[1]) & (data['Churn_Encode'] == 1)).sum()
non_churners1 = ((data['PaymentMethod']== payment_methods[1]) & (data['Churn_Encode']== 0)).sum()

churners2 = ((data['PaymentMethod'] == payment_methods[2]) & (data['Churn_Encode'] == 1)).sum()
non_churners2 = ((data['PaymentMethod']== payment_methods[2]) & (data['Churn_Encode']== 0)).sum()

churners3 = ((data['PaymentMethod'] == payment_methods[3]) & (data['Churn_Encode'] == 1)).sum()
non_churners3 = ((data['PaymentMethod']== payment_methods[3]) & (data['Churn_Encode']== 0)).sum()

In [ ]:
print(f"churners with payment method '{payment_methods[0]}': {churners}")
print(f"Non-Churners with payment method '{payment_methods[0]}': {non_churners}")

print(f"churners with payment method '{payment_methods[1]}': {churners1}")
print(f"Non-Churners with payment method '{payment_methods[1]}': {non_churners1}")

print(f"churners with payment method '{payment_methods[2]}': {churners2}")
print(f"Non-Churners with payment method '{payment_methods[2]}': {non_churners2}")

print(f"churners with payment method '{payment_methods[3]}': {churners3}")
print(f"Non-Churners with payment method '{payment_methods[3]}': {non_churners3}")

In [ ]:
# Load dataset
data = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')

In [ ]:
X = data.drop(['customerID', 'Churn'], axis=1)
y = data['Churn'].map({'Yes':1, 'No':0})

In [ ]:
# Preprocessing Pipeline
numerical_cols = ['tenure','MonthlyCharges','TotalCharges']
categorical_cols = [cols for cols in X.columns if cols not in numerical_cols]
preprocessor = ColumnTransformer([(
    'num',Pipeline([
        ('imputer',SimpleImputer(strategy='median')),
        ('scaler',StandardScaler())
    ]), numerical_cols,
     ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
)])

In [ ]:
# apply preprocessing
X_preprocessed = preprocessor.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_preprocessed, y, test_size=0.2, random_state=42)

In [ ]:
# Train Model
log_reg = LogisticRegression(random_state=42)
rf = RandomForestClassifier(random_state=42)
log_reg.fit(X_train, y_train)
rf.fit(X_train, y_train)

In [ ]:
# Evaluate Models
models = {"Logistic Regression": log_reg, "Random Forest": rf}
for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f"\n{name} performance:")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred):.4f}")
    print(f"ROC-AUC: {roc_auc_score(y_test, y_pred):.4f}")

In [ ]:
# Save RandomForest Model
joblib.dump(rf,'churn_model.pkl')
joblib.dump(preprocessor, "Preprocessor.pkl")

In [ ]:
# Feature importance for Random Forest
feature_names = numerical_cols + list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols))
importances = rf.feature_importances_
feature_importance = pd.Series(importances, index=feature_names).sort_values(ascending=False)
print("\nRandom Forest Feature Importance (Top 10):")
print(feature_importance.head(10))